In [ ]:
# ============================================================
# CELL 1 — Define Our Stock Watchlist
# ============================================================
# The "Magnificent 7" are the seven largest tech companies by
# market cap. We store their ticker symbols in a plain list.
# A ticker is just the short code a stock trades under on the
# exchange (e.g. AAPL = Apple, NVDA = Nvidia).

mag7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]

print("Tracking the following tickers:")
print(mag7)

In [ ]:
# ============================================================
# CELL 2 — Download 5 Years of Daily Price Data
# ============================================================
# yfinance is a free library that downloads stock data straight
# from Yahoo Finance. For each ticker we request OHLCV data:
#   Open   = price at market open
#   High   = highest price that day
#   Low    = lowest price that day
#   Close  = price at market close  ← this is what we'll chart
#   Volume = number of shares traded
#
# We store each ticker's data in a Python dictionary so we can
# look it up later with  data["AAPL"],  data["NVDA"], etc.

import yfinance as yf
import pandas as pd

data = {}   # empty dictionary — we'll fill it in the loop below

for ticker in mag7:
    print(f"Downloading {ticker}...")
    df = yf.download(ticker, period="5y", interval="1d", auto_adjust=True, progress=False)
    df.columns = df.columns.get_level_values(0)   # flatten multi-level column headers
    data[ticker] = df

print("\nAll downloads complete!")

In [ ]:
# ============================================================
# CELL 3 — Sanity Check: Preview First & Last 5 Rows
# ============================================================
# Before we trust any data we should always look at it.
# .head(5) shows the first 5 rows (oldest dates).
# .tail(5) shows the last 5 rows (most recent dates).
# If the dates and prices look reasonable, the download worked.

for ticker in mag7:
    print(f"\n{'='*55}")
    print(f"  {ticker}  —  {len(data[ticker])} trading days of data")
    print(f"{'='*55}")
    print("FIRST 5 ROWS (oldest):")
    print(data[ticker].head(5).to_string())
    print("\nLAST 5 ROWS (most recent):")
    print(data[ticker].tail(5).to_string())

In [ ]:
# ============================================================
# CELL 4 — Interactive Chart: All 7 Stocks, 5-Year Closing Price
# ============================================================
# Plotly creates interactive charts — you can zoom, pan, and
# hover over any point to see the exact date and price.
#
# Each stock gets its own colored line. The legend on the right
# acts as a toggle: click a stock name once to hide it, click
# again to bring it back. Double-click a name to isolate it.
#
# go.Figure()  = creates a blank chart canvas
# add_trace()  = adds one line (one stock) to the chart
# update_layout() = controls titles, axes, colors, and style

import plotly.graph_objects as go

# A nice color palette — one distinct color per stock
colors = {
    "AAPL":  "#A8E6CF",   # mint green
    "MSFT":  "#00BFFF",   # sky blue
    "GOOGL": "#FFD700",   # gold
    "AMZN":  "#FF8C00",   # orange
    "NVDA":  "#76EEC6",   # teal
    "META":  "#6495ED",   # cornflower blue
    "TSLA":  "#FF6B6B",   # coral red
}

fig = go.Figure()

for ticker in mag7:
    df = data[ticker]
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["Close"],
        mode="lines",
        name=ticker,
        line=dict(color=colors[ticker], width=1.8),
    ))

fig.update_layout(
    title=dict(
        text="Magnificent 7 — 5-Year Closing Prices",
        font=dict(size=22, color="white"),
        x=0.5,                      # center the title
    ),
    xaxis=dict(
        title="Date",
        title_font=dict(color="#aaaaaa"),
        tickfont=dict(color="#aaaaaa"),
        gridcolor="#2a2a2a",
        showline=True,
        linecolor="#444444",
    ),
    yaxis=dict(
        title="Closing Price (USD)",
        title_font=dict(color="#aaaaaa"),
        tickfont=dict(color="#aaaaaa"),
        tickprefix="$",
        gridcolor="#2a2a2a",
        showline=True,
        linecolor="#444444",
    ),
    legend=dict(
        title=dict(text="Click to toggle  ", font=dict(color="#aaaaaa")),
        font=dict(color="white"),
        bgcolor="#1a1a2e",
        bordercolor="#444444",
        borderwidth=1,
    ),
    paper_bgcolor="#0d0d1a",   # outer background
    plot_bgcolor="#111122",    # chart area background
    hovermode="x unified",     # shows all 7 prices when you hover on a date
    height=620,
)

fig.show()